# 59b. The two patterns of notebook 59, on fresh items

**Where this starts.** Notebook 59 closed on its own marks: sharedness (how alike the model's response to an instruction is in the document and in the user turn, beyond an ordinary sentence of about the same length) is real in all three undefended models (0.17, 0.30 and 0.24 above random weights, which give 0.00), but how much a defence changes it did not track how much the defence cuts attack success (rho +0.31, p 0.078), and the change in user drive did not track the change in user instructions followed (rho +0.20, p 0.087). Two patterns stood out that no mark had named, so nothing could be claimed from them:

1. Every trained defence that cuts attack success lowered sharedness: 15 of 16 changes below 0. The exception, twin distillation v1 (-0.008 [-0.022, +0.005]), is the defence that barely works (attack success down 0.067).
2. The ten defences trained with twin examples kept the user drive (changes -0.010 to +0.006); the six without them lost 0.12 to 0.30. Those six are the four trained without twins (the SecOPD loss, one of them with the literal phase added) and Meta-SecAlign-8B in both formats.

Patterns found by looking are claims only once they hold on items that played no part in finding them. This notebook writes both down as marks before reading anything, and tests them on fresh items: notebook 37's held-out Alpaca instructions (no adapter saw them) inside SQuAD validation passages (no adapter was trained on them), one item per passage. New instructions, new documents and a new kind of text; same models, same reader (`src/shared_drive.py`, unchanged), same five prompts per item (the clean passage; the instruction or an ordinary SQuAD sentence of about the same length on the passage's last line; the same two in the user turn), same window of five template positions and 16 shared answer tokens.

**One change from notebook 59.** These instructions have no task categories, so the user direction is built per half of the instructions instead of per left-out category: the items are split in two by instruction, every passage is used once, and each item is read against the direction built on the other half, which shares neither its passage nor its instruction.

**Marks, fixed in Stage A before any reading.**

| mark | what must hold |
|---|---|
| V | on each undefended model, sharedness on 100 fresh items at least 0.05 above the highest of three random-weight draws of its architecture. If it fails, the patterns cannot be read on these items. |
| C1: every working defence lowers sharedness | each of the 15 trained conditions whose attack success is at least 0.15 below its base model's (from the repository's tables, the paper's rule): change in sharedness with its interval below 0. All 15. |
| C2: the twins keep the user's instruction signal | each of the 10 twin-anchored conditions: change in user drive within 0.03 of 0; each of the 6 without the anchor (Qwen 3B distillation without twins at two scales and with the literal phase, Qwen 7B distillation without twins, Meta-SecAlign-8B in both formats): change at most -0.10 with its interval below 0. All 16. |
| reported | the weak defence outside C1; the two rank correlations of notebook 59 on these items; Meta-SecAlign's input role against the user turn; every change beside notebook 59's. |
| decision | V, C1 and C2 hold: CONFIRMED, both patterns hold on items notebook 59 never saw. One of C1 and C2 holds: PARTLY CONFIRMED, only that one can be claimed. Neither: NOT CONFIRMED, they were patterns of the BIPIA items. |

**Compute.** As notebook 59 with fewer random draws: 19 conditions times five prompts per item, three times one short generation per item for the answer tokens, two user directions per base model, and three random-weight draws of each architecture on 100 items. Every condition is cached as soon as it is read and the notebook resumes. The Hugging Face login and the tokenizer checks for Qwen 7B, Llama and Meta-SecAlign run at the end of Stage B.

**Outputs.** `reports/shared_drive/drive_59b.csv`, `statements_59b.csv`, `go_criteria_59b.csv`, `go_59b.txt`; `figures/shared_drive/confirm_59b.png`.

In [ ]:
# === SESSION BOOTSTRAP (run first): install from the local disk, then mount ===
!pip install -q transformers datasets accelerate sentencepiece bitsandbytes peft langdetect==1.0.9 scikit-learn
!pip uninstall -y -q torchao   # Colab ships an old torchao that the current PEFT rejects; nothing in this project uses it
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
for d_ in ('reports/shared_drive', 'figures/shared_drive', 'data/shared_drive_59b'): os.makedirs(d_, exist_ok=True)
import torch, gc
assert torch.cuda.is_available(), 'this notebook needs a GPU'
print('Session ready. GPU:', torch.cuda.get_device_name(0))

## Stage A. The conditions and what each achieves and costs (from the repository), the marks, the fresh items and their statements

In [ ]:
import importlib, hashlib, collections, numpy as np, json, re, pandas as pd
import task_state, head_patching, shared_drive, adaptive_attack, final_scoring, targets, model_defense, llama_h2h, fidelity_exact
for m_ in (task_state, head_patching, shared_drive, adaptive_attack, final_scoring, targets, model_defense, llama_h2h, fidelity_exact): importlib.reload(m_)
from task_state import suffix_ids
from head_patching import HeadHooks, capture_heads
from shared_drive import DriveHooks, read, encode_convs, ref_norms, item_drive, cluster_boot, pair_cosine, spearman_perm
from targets import load_lm, TASK_SYSTEM, TASK_USER
from model_defense import TASK_USER_WITH
from fidelity_exact import split_sentences, _clean_sentence_ok, norm
H = lambda s: hashlib.sha1((s or '').encode()).hexdigest()
def jload(f): return json.load(open(f)) if os.path.exists(f) else {}
def jsave(obj, f): json.dump(obj, open(f + '.tmp', 'w')); os.replace(f + '.tmp', f)
pd.set_option('display.width', 260); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 90)
OUT, REP, FIG = 'data/shared_drive_59b', 'reports/shared_drive', 'figures/shared_drive'
for d_ in (OUT, REP, FIG): os.makedirs(d_, exist_ok=True)

# ---- marks and settings, fixed before any result ----
SEED, BS, NANS, NBOOT, NPERM = 59, 16, 16, 5000, 20000
CRIT59 = dict(v_min=0.05, v_seeds=3, v_items=100, c1_gain=-0.15, c1_n=15, c2_twin_abs=0.03, c2_notwin_max=-0.10, h1_n=16, h2_n=15, min_items=150)
print('marks:', CRIT59)
# the two patterns of notebook 59, written down before any reading of the fresh items:
#   C1: every trained defence that cuts attack success by at least 0.15 lowers sharedness (its change's interval below 0)
#   C2: the defences trained with twin examples keep the user drive (change within 0.03 of 0), and the six without them lose at least 0.10 (interval below 0)
TWIN = ('q3_v1', 'q3_v2', 'q3_v3', 'q3_v4', 'q3_replay', 'q3_stlit', 'q7_v4', 'q7_v5', 'l_p1', 'l_full')       # trained with twins (the user-turn anchor)
NOTWIN = ('q3_notwin1', 'q3_notwin2', 'q3_ntlit', 'q7_notwin', 'l_sec', 'l_sec_user')                             # trained without them: the SecOPD loss alone, and Meta-SecAlign

# ---- the 19 conditions: weights, prompt format, and where their attack rate, user measure and deletion gap are in the repository ----
QWEN3B, QWEN7B, SEC = 'Qwen/Qwen2.5-3B-Instruct', 'Qwen/Qwen2.5-7B-Instruct', llama_h2h.SECALIGN
FT, CS, CU, AA, U7, LU = 'reports/paper_fixed_all_models.csv', 'reports/controls_security.csv', 'reports/controls_utility.csv', 'reports/ablation_anchor_scale.csv', 'reports/fidelity_v5_7b_utility.csv', 'reports/llama_utility.csv'
CF, F7, LF = 'reports/controls_fidelity.csv', 'reports/fidelity_v5_7b_fidelity.csv', 'reports/llama_fidelity.csv'
def C_(tag, fam, adapter, label, asr, uf, dg, mode='plain', tokf=None): return dict(tag=tag, fam=fam, adapter=adapter, label=label, asr=asr, uf=uf, dg=dg, mode=mode, tokf=tokf)
CONDS = [
    C_('q3_undef',   'q3', None, 'Qwen 3B undefended', (FT, 'Qwen 3B undefended'), (CU, 'Qwen 3B undefended'), (CF, 'Qwen 3B undefended')),
    C_('q3_v1',      'q3', 'data/distill/lora_3b', 'Qwen 3B twin distillation v1', (FT, 'twin_distilled_v1'), (AA, 'twins, v1 scale (1,500 attacked + 1,500 twins)'), None),
    C_('q3_notwin1', 'q3', 'data/distill_notwin/lora_3b', 'Qwen 3B distillation without twins, v1 scale', (FT, 'no_twin'), (AA, 'no twin, v1 scale (1,500 attacked)'), None),
    C_('q3_notwin2', 'q3', 'data/distill_notwin2/lora_3b', 'Qwen 3B distillation without twins, v2 scale', (FT, 'no_twin_v2'), (CU, 'no twins (v2 scale)'), (CF, 'no twins (v2 scale)')),
    C_('q3_v2',      'q3', 'data/distill2/lora_3b', 'Qwen 3B twin distillation v2', (FT, 'Qwen 3B twin distillation v2'), (CU, 'v2 (twin distillation)'), (CF, 'v2 (twin distillation)')),
    C_('q3_v3',      'q3', 'data/distill3/lora_3b', 'Qwen 3B twin distillation v3', (FT, 'twin_distilled_v3'), None, None),
    C_('q3_v4',      'q3', 'data/distill4/lora_3b', 'Qwen 3B full recipe (v4)', (FT, 'Qwen 3B full recipe (final)'), (CU, 'v4 = v2 + literal phase (final)'), (CF, 'v4 = v2 + literal phase (final)')),
    C_('q3_ntlit',   'q3', 'data/controls/notwin_literal/lora_3b', 'Qwen 3B no twins + literal phase', (CS, 'no twins + literal phase'), (CU, 'no twins + literal phase'), (CF, 'no twins + literal phase')),
    C_('q3_replay',  'q3', 'data/controls/replay_only/lora_3b', 'Qwen 3B v2 + replay only', (CS, 'v2 + replay only (equal budget)'), (CU, 'v2 + replay only (equal budget)'), (CF, 'v2 + replay only (equal budget)')),
    C_('q3_stlit',   'q3', 'data/controls/statement_literal/lora_3b', 'Qwen 3B v2 + statement-only literal phase', (CS, 'v2 + literal phase, statements only'), (CU, 'v2 + literal phase, statements only'), (CF, 'v2 + literal phase, statements only')),
    C_('q7_undef',   'q7', None, 'Qwen 7B undefended', (FT, 'Qwen 7B undefended'), (U7, 'undefended 7B'), (F7, 'undefended_7b')),
    C_('q7_notwin',  'q7', 'data/distill7b/lora_7b', 'Qwen 7B distillation without twins', (FT, 'twin_distilled_7b'), (U7, 'distilled 7B (no twins)'), (F7, 'twin_distilled_7b')),
    C_('q7_v4',      'q7', 'data/distill7b_v4/lora_7b', 'Qwen 7B twins + literal (v4)', (FT, 'fidelity_v4_7b'), (U7, 'fidelity_v4_7b (twins + literal)'), (F7, 'fidelity_v4_7b')),
    C_('q7_v5',      'q7', 'data/distill7b_v5/lora_7b', 'Qwen 7B full recipe (v5)', (FT, 'Qwen 7B full recipe (final)'), (U7, 'fidelity_v5_7b (rebalanced toward the defense)'), (F7, 'fidelity_v5_7b')),
    C_('l_undef',    'll', None, 'Llama 8B undefended', (FT, 'Llama 8B undefended'), (LU, 'undefended Llama-3.1-8B'), (LF, 'undefended Llama-3.1-8B')),
    C_('l_p1',       'll', 'data/llama/phase1/lora', 'Llama 8B ours, phase 1', (FT, 'llama_twin_p1'), (LU, 'ours, phase 1 (twin distillation)'), (LF, 'ours, phase 1 (twin distillation)')),
    C_('l_full',     'll', 'data/llama/phase2/lora', 'Llama 8B ours, full recipe', (FT, 'Llama 8B ours, full recipe'), (LU, 'ours, full recipe'), (LF, 'ours, full recipe')),
    C_('l_sec',      'll', SEC, 'Meta-SecAlign-8B, input role (as specified)', (FT, 'Meta-SecAlign-8B (as specified)'), (LU, 'Meta-SecAlign-8B (as specified)'), (LF, 'Meta-SecAlign-8B (as specified)'), mode='input_role', tokf=SEC),
    C_('l_sec_user', 'll', SEC, 'Meta-SecAlign-8B, document in the user turn', (FT, 'Meta-SecAlign-8B (data in the user turn)'), (LU, 'Meta-SecAlign-8B (data in the user turn)'), (LF, 'Meta-SecAlign-8B (data in the user turn)'), mode='plain', tokf=SEC)]
CD = {c['tag']: c for c in CONDS}; assert len(CD) == 19
FAM = {'q3': dict(base=QWEN3B, label='Qwen 3B', llama=False), 'q7': dict(base=QWEN7B, label='Qwen 7B', llama=False), 'll': dict(base=llama_h2h.BASE, label='Llama 8B', llama=True)}
BASE_OF = {f: next(c['tag'] for c in CONDS if c['fam'] == f and c['adapter'] is None) for f in FAM}
for c in CONDS:
    if c['adapter'] and c['adapter'] != SEC: assert os.path.exists(os.path.join(c['adapter'], 'adapter_config.json')), f"missing adapter {c['adapter']}"

# ---- what each condition achieves and costs, read from the repository's tables ----
def num(s):
    m_ = re.match(r'\s*([+-]?\d*\.?\d+)', str(s)); return float(m_.group(1)) if m_ else np.nan
def cell(path, label, col):
    T_ = pd.read_csv(path); r_ = T_[T_.iloc[:, 0] == label]; assert len(r_) == 1, f'{path}: {len(r_)} rows named {label!r}'; return num(r_[col].iloc[0])
def delgap(path, label):
    T_ = pd.read_csv(path); r_ = T_[T_.iloc[:, 0] == label]
    assert len(r_) == 3 and set(r_.task) == {'repeat', 'number', 'quote_last'}, f'{path}: rows for {label!r}'
    return float(np.mean([num(x) for x in r_.deletion_gap]))
COST = {c['tag']: dict(asr=cell(c['asr'][0], c['asr'][1], 'fixed_13') if c['asr'] else np.nan,
                       uf=cell(c['uf'][0], c['uf'][1], 'user_followed_150') if c['uf'] else np.nan,
                       dg=delgap(*c['dg']) if c['dg'] else np.nan) for c in CONDS}
assert cell(AA, 'undefended', 'user_followed_150') == COST['q3_undef']['uf'], 'the two 3B user-following tables disagree on the undefended model'
assert all(cell(CS, a_, 'fixed_13') == cell(FT, b_, 'fixed_13') for a_, b_ in (('Qwen 3B undefended', 'Qwen 3B undefended'), ('v2 (twin distillation)', 'Qwen 3B twin distillation v2'), ('v4 = v2 + literal phase (final)', 'Qwen 3B full recipe (final)'), ('no twins (v2 scale)', 'no_twin_v2'))), 'the controls\' attack table and the paper\'s disagree on the models they share'
for c in CONDS:
    b_ = BASE_OF[c['fam']]; COST[c['tag']].update(d_asr=COST[c['tag']]['asr'] - COST[b_]['asr'], d_uf=COST[c['tag']]['uf'] - COST[b_]['uf'])
CT = pd.DataFrame([dict(condition=c['tag'], label=c['label'], **{k: round(v, 3) for k, v in COST[c['tag']].items()}) for c in CONDS])
print('\n=== what each condition achieves and costs (from the repository, the paper\'s rule; d_ = minus its own undefended model) ==='); print(CT.to_string(index=False))
H1_SET = [c['tag'] for c in CONDS if c['adapter'] and COST[c['tag']]['asr'] == COST[c['tag']]['asr']]
H2_SET = [c['tag'] for c in CONDS if c['adapter'] and COST[c['tag']]['uf'] == COST[c['tag']]['uf']]
assert len(H1_SET) == CRIT59['h1_n'] and len(H2_SET) == CRIT59['h2_n'], (len(H1_SET), len(H2_SET))
assert set(TWIN) | set(NOTWIN) == {c['tag'] for c in CONDS if c['adapter']} and not set(TWIN) & set(NOTWIN)
C1_SET = [t for t in H1_SET if COST[t]['d_asr'] <= CRIT59['c1_gain']]; assert len(C1_SET) == CRIT59['c1_n'], C1_SET
print('C1 applies to', len(C1_SET), 'conditions (attack success down by at least 0.15); not to', sorted(set(H1_SET) - set(C1_SET)))

# ---- the fresh items: notebook 37's held-out Alpaca instructions (no adapter saw them) and SQuAD validation passages (no adapter trained on them) ----
FI = json.load(open('data/fidelity_exact/items.json')); seen_, ITEMS = set(), []
for it in FI:
    if it['passage'] in seen_: continue
    seen_.add(it['passage']); ITEMS.append(dict(passage=int(it['passage']), doc=' '.join(it['base']), base0=it['base'][0], instruction=it['instruction'].strip()))
n = len(ITEMS); assert n >= CRIT59['min_items'], f'only {n} distinct passages'
CLEAN_DOC = [it['doc'] for it in ITEMS]; INS = [it['instruction'] for it in ITEMS]
SUBTEXT = [d + '\n' + x for d, x in zip(CLEAN_DOC, INS)]                                  # the instruction on the document's last line, as in the BIPIA items
HOSTS = CLEAN_DOC; HOST = np.arange(n)                                                   # one item per passage: each item is its own host
print(f'fresh items: {n} (distinct passages {n}, distinct instructions {len(set(INS))})')
_IMP = re.compile(r"^(please|ignore|write|translate|tell|say|give|make|list|summari[sz]e|provide|include|add|use|do|don't|let|answer|respond|reply|describe|explain|create|send|click|visit|check|find|call|contact|read|follow|replace|convert|encode|decode|reverse|rewrite|output|print|start|forget|disregard|remember|note|keep|stop|try|go|see|share|show|change|put|insert|append|repeat|copy|type|enter|return|consider|imagine)\b", re.I)
def ordinary(s): return _clean_sentence_ok(s, 4, 40) and s.endswith('.') and '\n' not in s and not _IMP.match(s) and not re.search(r"\b(please|you|your)\b", s, re.I)
# the ordinary sentences come from SQuAD validation paragraphs (Wikipedia), the source of notebook 37's statements; the BIPIA hosts hold too few clean prose sentences
try:
    from datasets import load_dataset
    SRC_TXT = sorted(set(load_dataset('rajpurkar/squad', split='validation')['context'])); SRC_NAME = 'SQuAD validation paragraphs'
except Exception as e_:
    print('SQuAD could not be loaded (' + type(e_).__name__ + '); falling back to notebook 37\'s saved statements')
    SRC_TXT = sorted({it['statement'] for it in json.load(open('data/fidelity_exact/items.json'))}); SRC_NAME = "notebook 37's statements"
POOL = sorted({s for d in SRC_TXT for s in split_sentences(d) if ordinary(s)})
SENT_SRC = {norm(x): k for k, d in enumerate(SRC_TXT) for x in split_sentences(d)}         # sentence -> its paragraph, so a statement never comes from the item's own paragraph
PARA = [SENT_SRC.get(norm(it['base0'])) for it in ITEMS]; POOL_PARA = np.array([SENT_SRC.get(norm(x), -1) for x in POOL])
print(f'items whose SQuAD paragraph was found: {sum(p_ is not None for p_ in PARA)} of {n}')
print(f'\nordinary sentences available: {len(POOL)} from {SRC_NAME}')
assert len(POOL) >= 200, 'too few ordinary sentences to build the statement control'
wc = lambda s: len(s.split())
POOL_WC = np.array([wc(s) for s in POOL]); POOL_N = [norm(s) for s in POOL]; STMT = []
for i in range(n):
    gap = np.abs(POOL_WC - wc(INS[i])); order_ = np.lexsort((np.random.default_rng(SEED + i).random(len(POOL)), gap))      # nearest in length first, ties in a seeded random order
    cand = [POOL[j] for j in order_[:60] if POOL_N[j] not in norm(CLEAN_DOC[i]) and (PARA[i] is None or POOL_PARA[j] != PARA[i])][:10]
    assert cand, f'no statement for item {i}'
    STMT.append(cand[int(np.random.default_rng(SEED + i).integers(len(cand)))])
ST = pd.DataFrame(dict(item=range(n), passage=[it['passage'] for it in ITEMS], instruction=INS, statement=STMT, words_instruction=[wc(x) for x in INS], words_statement=[wc(x) for x in STMT]))
ST.to_csv(f'{REP}/statements_59b.csv', index=False)
print(f'words, median: instruction {int(ST.words_instruction.median())}, statement {int(ST.words_statement.median())} | distinct statements used {ST.statement.nunique()} | most uses of one statement {ST.statement.value_counts().iloc[0]}')
print(ST.sample(4, random_state=SEED)[['instruction', 'statement']].to_string(index=False))

# ---- two halves of the items by instruction; every passage is used once, so the halves share neither a passage nor an instruction, and an item is read against the user direction of the other half ----
UI = sorted(set(INS)); perm_ = np.random.default_rng(SEED).permutation(len(UI)); FOLD_OF_I = {UI[j]: int(k % 2) for k, j in enumerate(perm_)}
FOLD = np.array([FOLD_OF_I[x] for x in INS]); REF_OF = 1 - FOLD; NREF = 2
print('items per half:', int((FOLD == 0).sum()), int((FOLD == 1).sum())); assert min((FOLD == 0).sum(), (FOLD == 1).sum()) >= 60

# ---- the five prompts per item, read together (one item's five prompts always share a forward pass) ----
# each prompt is kept as (task part, document): the plain format joins them with a blank line, exactly as TASK_USER and TASK_USER_WITH do; the input-role
# format puts the task part in the user role and the document in the input role, as llama_h2h.messages_for does, without relying on a blank-line split
T_HEAD, TW_HEAD = TASK_USER.split('\n\n{doc}')[0], TASK_USER_WITH.split('\n\n{doc}')[0]
assert TASK_USER == T_HEAD + '\n\n{doc}' and TASK_USER_WITH == TW_HEAD + '\n\n{doc}' and '{doc}' not in T_HEAD + TW_HEAD
TYPES = ('P', 'D_ins', 'D_stmt', 'U_ins', 'U_stmt')
PART = {'P': [(T_HEAD, CLEAN_DOC[i]) for i in range(n)],
        'D_ins': [(T_HEAD, SUBTEXT[i]) for i in range(n)],
        'D_stmt': [(T_HEAD, CLEAN_DOC[i] + '\n' + STMT[i]) for i in range(n)],
        'U_ins': [(TW_HEAD.format(instr=INS[i]), CLEAN_DOC[i]) for i in range(n)],
        'U_stmt': [(TW_HEAD.format(instr=STMT[i]), CLEAN_DOC[i]) for i in range(n)]}
PROMPT = {t: [h_ + '\n\n' + d_ for h_, d_ in PART[t]] for t in TYPES}
assert PROMPT['P'][0] == TASK_USER.format(doc=CLEAN_DOC[0]) and PROMPT['U_ins'][0] == TASK_USER_WITH.format(instr=INS[0], doc=CLEAN_DOC[0])
def conv(t, i, mode):
    h_, d_ = PART[t][i]; sys_ = [{'role': 'system', 'content': TASK_SYSTEM}]
    if mode == 'plain': return sys_ + [{'role': 'user', 'content': h_ + '\n\n' + d_}]
    if d_.startswith('Input:\n'): d_ = d_[len('Input:\n'):]
    return sys_ + [{'role': 'user', 'content': h_}, {'role': 'input', 'content': llama_h2h.recursive_filter(d_)}]
NBL = sum('\n\n' in h_ for t in TYPES for h_, _ in PART[t])
print(f'task parts holding a blank line (an instruction of several paragraphs): {NBL}; the input-role prompts are built from the parts, so these keep the whole instruction in the user role')
for mode in ('plain', 'input_role'):                                                  # where the project's own helper splits correctly, the prompts are identical to it
    same = [conv(t, i, mode) == llama_h2h.messages_for(PROMPT[t][i], mode, TASK_SYSTEM) for t in TYPES for i in range(n) if '\n\n' not in PART[t][i][0]]
    assert all(same), f'{mode}: {len(same) - sum(same)} prompts differ from llama_h2h.messages_for'
ROWS = [(t, i) for t in TYPES for i in range(n)]; RIX = {ri: r for r, ri in enumerate(ROWS)}; GROUP = [(t, int(FOLD[i])) for t, i in ROWS]
ROWS_OF = {t: np.array([RIX[(t, i)] for i in range(n)]) for t in TYPES}
PAIRS = [(RIX[('D_ins', i)], RIX[('U_ins', i)], RIX[('P', i)], RIX[('P', i)]) for i in range(n)] + [(RIX[('D_stmt', i)], RIX[('U_stmt', i)], RIX[('P', i)], RIX[('P', i)]) for i in range(n)]
ITEM_ORDER = sorted(range(n), key=lambda i: (len(CLEAN_DOC[i]), i)); BI = 4
BATCH5 = [[RIX[(t, i)] for i in ITEM_ORDER[s_: s_ + BI] for t in TYPES] for s_ in range(0, n, BI)]
V_ITEMS = [ITEM_ORDER[int(k)] for k in np.unique(np.linspace(0, n - 1, min(CRIT59['v_items'], n)).round())]; assert len(V_ITEMS) == min(CRIT59['v_items'], n)   # the items the random-weight readings use, spread over the lengths
PH = H(json.dumps([PROMPT[t][i] for t, i in ROWS]))[:12]
def convs_of(mode): return [conv(t, i, mode) for t, i in ROWS]
print(f'prompts: {len(ROWS)} per condition in {len(BATCH5)} passes of {BI} items (hash {PH})')
assert PROMPT['D_ins'][0].endswith('\n' + INS[0]) and PROMPT['D_stmt'][0].endswith('\n' + STMT[0])

## Stage B. Loading, the shared answer tokens, and the reader's self-test on the 3B base model

In [ ]:
from peft import PeftModel
M = dict(model=None, tok=None, tag=None, fam=None)
def free_model():
    M.update(model=None, tok=None, tag=None, fam=None); gc.collect(); torch.cuda.empty_cache()
def load_cond(tag):
    """The weights and tokenizer of a condition exactly as notebooks 30 to 45 loaded them."""
    c = CD[tag]; f = FAM[c['fam']]
    if M['tag'] == tag or (M['model'] is not None and M['tag'] in CD and M['fam'] == c['fam'] and CD[M['tag']]['adapter'] == c['adapter'] and CD[M['tag']]['tokf'] == c['tokf']):
        M['tag'] = tag; return M['model'], M['tok']                                      # the same weights in another prompt format (Meta-SecAlign)
    free_model()
    if f['llama']: m_, t_ = llama_h2h.load_llama(adapter=c['adapter'], tokenizer_from=c['tokf'])
    else:
        m_, t_ = load_lm(f['base'], four_bit=True)
        if c['adapter']: m_ = PeftModel.from_pretrained(m_, c['adapter']).eval()
    M.update(model=m_, tok=t_, tag=tag, fam=c['fam']); print(f'loaded {c["label"]}' + (f' ({c["adapter"]})' if c['adapter'] else '')); return m_, t_
def load_random(fam, seed):
    """The family's architecture with random weights (seeded), in bf16: one draw of the null for sharedness. Same tokenizer, same prompts, same answer tokens."""
    from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
    free_model(); cfg = AutoConfig.from_pretrained(FAM[fam]['base']); torch.manual_seed(1000 * SEED + seed)
    with torch.device('cuda' if torch.cuda.is_available() else 'cpu'): m_ = AutoModelForCausalLM.from_config(cfg, torch_dtype=torch.bfloat16).eval()
    t_ = AutoTokenizer.from_pretrained(FAM[fam]['base'])
    if t_.pad_token is None: t_.pad_token = t_.eos_token
    M.update(model=m_, tok=t_, tag=f'{fam}_random{seed}', fam=fam); return m_, t_
def answers_for(fam):
    """16 answer tokens per item: the base model's greedy start of an answer to D_stmt, generated once and shared by
    every prompt of the item and every condition of the family."""
    fp = f'{OUT}/answers_{fam}.json'; A_ = jload(fp)
    if A_.get('ph') == PH and len(A_.get('ids', [])) == n and all(len(a_) == NANS for a_ in A_['ids']): return A_
    m_, t_ = load_cond(BASE_OF[fam]); sfx = suffix_ids(t_, TASK_SYSTEM)
    cv = [conv('D_stmt', i, 'plain') for i in range(n)]; ids_ = [None] * n
    for s in range(0, n, BS):
        ch = list(range(s, min(n, s + BS))); ii, am = encode_convs(t_, [cv[i] for i in ch], sfx); dev = m_.get_input_embeddings().weight.device
        with torch.no_grad():
            g = m_.generate(input_ids=ii.to(dev), attention_mask=am.to(dev), max_new_tokens=NANS, min_new_tokens=NANS, do_sample=False, temperature=None, top_p=None, pad_token_id=t_.pad_token_id)
        for j, i in enumerate(ch): ids_[i] = [int(x) for x in g[j, ii.shape[1]:]]
    assert all(len(a_) == NANS for a_ in ids_), 'an answer start is shorter than 16 tokens'
    A_ = dict(ph=PH, vocab=len(t_), ids=ids_, text=[t_.decode(a_, skip_special_tokens=True) for a_ in ids_]); jsave(A_, fp)
    print(f'{FAM[fam]["label"]}: answer starts generated, e.g. {A_["text"][0]!r}'); return A_

# ---- the self-test, on the 3B base model ----
m, tok = load_cond('q3_undef'); SFX3 = suffix_ids(tok, TASK_SYSTEM); K = len(SFX3); W = K + NANS
ANS = {'q3': answers_for('q3')}
def self_test():
    its = ITEM_ORDER[:2]; rows = [RIX[(t, i)] for i in its for t in TYPES]; loc = {r: j for j, r in enumerate(rows)}
    cv = convs_of('plain'); sc = [cv[r] for r in rows]; sg = [GROUP[r] for r in rows]; sa = [ANS['q3']['ids'][ROWS[r][1]] for r in rows]
    sp = [tuple(loc[x] for x in PAIRS[q]) for q in (its[0], its[1], n + its[0], n + its[1])]
    hk = DriveHooks(m, W, K); Rr = torch.randn(2, hk.L, W, hk.D, generator=torch.Generator().manual_seed(SEED)); hk.set_refs(Rr)
    pr, cnt, raw, pdo = read(m, hk, tok, sc, sg, SFX3, ans_ids=sa, keep_raw=True, batches=[list(range(len(rows)))], pairs=sp)
    man = np.stack([np.stack([np.stack([np.einsum('kd,rkd->r', raw[j][l][:K].numpy(), Rr[:, l, :K].numpy()), np.einsum('kd,rkd->r', raw[j][l][K:].numpy(), Rr[:, l, K:].numpy())], -1) for l in range(hk.L)]) for j in range(len(rows))])
    e_proj = float(np.abs(man - pr).max() / np.abs(man).max())
    e_pair = 0.0
    for q, (a_, b_, c_, d_) in enumerate(sp):
        A_ = (raw[a_] - raw[c_]).flatten(); B__ = (raw[b_] - raw[d_]).flatten()
        e_pair = max(e_pair, abs(float(pair_cosine(pdo[q:q + 1])[0][0]) - float(A_ @ B__ / (A_.norm() * B__.norm()))))
    _, _, raw1, _ = read(m, hk, tok, sc[2:3], sg[2:3], SFX3, ans_ids=sa[2:3], batch=1, keep_raw=True)
    cosf = lambda a_, b_: float((a_.flatten() * b_.flatten()).sum() / (a_.norm() * b_.norm()))
    c_alone = cosf(raw1[0], raw[2])
    hh = HeadHooks(m, K); cap = capture_heads(m, hh, tok, [PROMPT[ROWS[r][0]][ROWS[r][1]] for r in rows], TASK_SYSTEM, SFX3); hh.close()
    c_cap = min(cosf(cap[j].float(), raw[j][:, :K]) for j in range(len(rows)))
    hk.close()
    print(f'self-test on {CD["q3_undef"]["label"]}: layers {hk.L}, heads {hk.nh} of size {hk.dh}, window {W} ({K} template positions + {NANS} answer tokens); '
          f'projections against a direct computation {e_proj:.1e}; item-level cosines against a direct computation {e_pair:.1e}; one row read alone against the same row in a batch of ten, cosine {c_alone:.6f}; '
          f'template part against the reading of notebooks 56 to 58 (the prompt pass of generate), lowest cosine {c_cap:.6f}')
    assert e_proj < 1e-3 and e_pair < 1e-4 and c_alone > 0.999 and c_cap > 0.999, 'the reader does not behave on this model: stop'
self_test()
del m                                                                                # the model stays in M for Stage C; this name must not pin it on the GPU

# ---- the checks that need only tokenizers, done now rather than hours into Stage C ----
from transformers import AutoTokenizer
for f in ('q7', 'll'):
    if FAM[f]['llama']: llama_h2h.hf_login()                                         # Llama and Meta-SecAlign are gated: the Colab secret HF_TOKEN, or a hidden box here
    tf_ = AutoTokenizer.from_pretrained(FAM[f]['base']); assert len(suffix_ids(tf_, TASK_SYSTEM)) == K, f'{f}: the template suffix is not {K} tokens'
st_ = AutoTokenizer.from_pretrained(SEC); llama_h2h.check_input_role_template(st_)
if st_.pad_token is None: st_.pad_token = st_.eos_token
encode_convs(st_, [conv('U_ins', i, 'input_role') for i in range(3)], suffix_ids(tf_, TASK_SYSTEM))   # fails here if the input role ends differently
print('tokenizer checks passed: Qwen 7B and Llama have the same', K, 'template positions; Meta-SecAlign\'s input role ends with the base template suffix')
del tf_, st_

## Stage C. Every condition read: one forward pass per prompt, cached per condition

In [ ]:
def ref_for(fam, sfx):
    """The family's user direction: in the undefended base model, U_ins minus U_stmt, one mean per half of the instructions."""
    A_ = ANS[fam]; key = f'{PH}|{H(json.dumps(A_["ids"]))[:12]}|{W}'; fp = f'{OUT}/ref_{fam}.pt'
    if os.path.exists(fp):
        d_ = torch.load(fp, weights_only=False)
        if d_.get('key') == key: return d_['u'], key
    m_, t_ = load_cond(BASE_OF[fam]); rows = [RIX[(t, i)] for i in ITEM_ORDER for t in ('U_ins', 'U_stmt')]; cv = convs_of('plain'); sub_c = [cv[r] for r in rows]
    hk = DriveHooks(m_, W, K)
    _, cnt, _, _ = read(m_, hk, t_, sub_c, [GROUP[r] for r in rows], sfx, ans_ids=[A_['ids'][ROWS[r][1]] for r in rows], batch=BS)
    S_ = {g: s_.cpu() for g, s_ in hk.sums.items()}; hk.close()
    u = torch.stack([S_[('U_ins', f)] / cnt[('U_ins', f)] - S_[('U_stmt', f)] / cnt[('U_stmt', f)] for f in (0, 1)])     # [2, L, W, D], one per half of the instructions
    torch.save(dict(key=key, u=u), fp); print(f'{FAM[fam]["label"]}: {len(u)} user directions built ({len(rows)} prompts)'); return u, key
def run_random(fam, seed, rkey, sfx):
    """One random-weight draw of the family's architecture, read on the V items: the item-level cosines only."""
    tag = f'{fam}_random{seed}'; key = f'{rkey}|random|{SEED}|{seed}|{H(json.dumps(V_ITEMS))[:8]}'; fp = f'{OUT}/{tag}.pt'
    if os.path.exists(fp):
        d_ = torch.load(fp, weights_only=False)
        if d_.get('key') == key: return d_
    m_, t_ = load_random(fam, seed); assert suffix_ids(t_, TASK_SYSTEM) == sfx
    rows = [RIX[(t, i)] for i in V_ITEMS for t in TYPES]; loc = {r: j for j, r in enumerate(rows)}; cv = convs_of('plain')
    prs = [tuple(loc[x] for x in PAIRS[i]) for i in V_ITEMS] + [tuple(loc[x] for x in PAIRS[n + i]) for i in V_ITEMS]
    hk = DriveHooks(m_, W, K)
    _, _, _, pdo = read(m_, hk, t_, [cv[r] for r in rows], [GROUP[r] for r in rows], sfx, ans_ids=[ANS[fam]['ids'][ROWS[r][1]] for r in rows],
                        batches=[list(range(s_, min(len(rows), s_ + BI * len(TYPES)))) for s_ in range(0, len(rows), BI * len(TYPES))], pairs=prs)
    hk.close(); free_model(); d_ = dict(key=key, pairs=pdo); torch.save(d_, fp); print(f'  {FAM[fam]["label"]} with random weights, draw {seed}: read, {len(rows)} prompts'); return d_
def run_cond(tag, u, rkey, sfx):
    c = CD[tag]; key = f'{rkey}|{c["adapter"]}|{c["mode"]}|{c["tokf"]}'; fp = f'{OUT}/{tag}.pt'
    if os.path.exists(fp):
        d_ = torch.load(fp, weights_only=False)
        if d_.get('key') == key: return d_
    m_, t_ = load_cond(tag)
    assert suffix_ids(t_, TASK_SYSTEM) == sfx, f'{tag}: the template suffix differs from the base model\'s'
    assert [t_.decode(a_, skip_special_tokens=True) for a_ in ANS[c['fam']]['ids'][:20]] == ANS[c['fam']]['text'][:20], f'{tag}: this tokenizer reads the shared answer tokens differently from the base model\'s'
    cv = convs_of(c['mode']); hk = DriveHooks(m_, W, K); hk.set_refs(u)
    proj, cnt, _, pdo = read(m_, hk, t_, cv, GROUP, sfx, ans_ids=[ANS[c['fam']]['ids'][i] for _, i in ROWS], batches=BATCH5, pairs=PAIRS)
    d_ = dict(key=key, proj=proj, pairs=pdo, cnt=cnt, nh=hk.nh, L=hk.L)
    hk.close(); torch.save(d_, fp); print(f'  {c["label"]}: read, {len(ROWS)} prompts'); return d_

RES, U_REF, NORMS, SFX = {}, {}, {}, {}
for fam in ('q3', 'q7', 'll'):
    tags = [c['tag'] for c in CONDS if c['fam'] == fam]
    if fam not in ANS: ANS[fam] = answers_for(fam)
    t0_ = load_cond(BASE_OF[fam])[1] if not all(os.path.exists(f'{OUT}/{t}.pt') for t in tags) else None
    if t0_ is None:
        from transformers import AutoTokenizer
        t0_ = AutoTokenizer.from_pretrained(FAM[fam]['base'])
    SFX[fam] = suffix_ids(t0_, TASK_SYSTEM); assert len(SFX[fam]) == K, f'{fam}: {len(SFX[fam])} template positions, not {K}'
    U_REF[fam], rkey = ref_for(fam, SFX[fam]); NORMS[fam] = ref_norms(U_REF[fam].numpy(), K)
    print(f'{FAM[fam]["label"]}: squared size of the {len(NORMS[fam])} user directions, template part {NORMS[fam][:, 0].min():.3g} to {NORMS[fam][:, 0].max():.3g}, answer part {NORMS[fam][:, 1].min():.3g} to {NORMS[fam][:, 1].max():.3g}')
    for t in tags:
        RES[t] = run_cond(t, U_REF[fam], rkey, SFX[fam])
        if t != BASE_OF[fam]:
            d_ = np.abs(RES[t]['proj'] - RES[BASE_OF[fam]]['proj']).max(); s_ = np.abs(RES[BASE_OF[fam]]['proj']).max()
            assert d_ > 1e-3 * s_, f'{t}: its readings equal the base model\'s, so the adapter is not active'
    for sd_ in range(CRIT59['v_seeds']): RES[f'{fam}_random{sd_}'] = run_random(fam, sd_, rkey, SFX[fam])
free_model()
print(f'\nall 19 conditions and {3 * CRIT59["v_seeds"]} random-weight readings done')

## Stage D. The drives, the marks and what they decide

In [ ]:
import matplotlib.pyplot as plt
from reslog import log_result
fmt = lambda t3: f'{t3[0]:+.3f} [{t3[1]:+.3f}, {t3[2]:+.3f}]'
DR = {}
for t, fam_ in [(c['tag'], c['fam']) for c in CONDS]:
    P_ = RES[t]['proj']; PD_ = RES[t]['pairs']; nr = NORMS[fam_]; d_ = {}
    ci_, _ = pair_cosine(PD_[:n]); cs_, _ = pair_cosine(PD_[n:])
    d_['cos_ins'], d_['cos_stmt'], d_['shared'] = ci_, cs_, ci_ - cs_
    d_['usr'], _ = item_drive(P_[ROWS_OF['U_ins']], P_[ROWS_OF['U_stmt']], REF_OF, nr, 'window')
    d_['inj'], _ = item_drive(P_[ROWS_OF['D_ins']], P_[ROWS_OF['D_stmt']], REF_OF, nr, 'window')
    DR[t] = d_
B_ = {}
for c in CONDS:
    t = c['tag']; b_ = BASE_OF[c['fam']]; bt = lambda k: cluster_boot(DR[t][k], HOST, NBOOT, SEED)
    ch = lambda k: cluster_boot(DR[t][k] - DR[b_][k], HOST, NBOOT, SEED) if t != b_ else (0.0, 0.0, 0.0)
    B_[t] = dict(shared=bt('shared'), usr=bt('usr'), inj=bt('inj'), dshared=ch('shared'), dusr=ch('usr'), dinj=ch('inj'))
# notebook 59 beside it, read from its committed table
N59 = pd.read_csv(f'{REP}/drive.csv').set_index('condition')
p59 = lambda t, col: num(N59.loc[t, col]) if t in N59.index and isinstance(N59.loc[t, col], str) else np.nan
rows = []
for c in CONDS:
    t = c['tag']; b_ = BASE_OF[c['fam']]
    rows.append(dict(condition=t, label=c['label'], recipe='base' if t == b_ else ('twins' if t in TWIN else 'no twins'), sharedness=fmt(B_[t]['shared']),
                     cos_instruction=round(float(DR[t]['cos_ins'].mean()), 3), cos_statement=round(float(DR[t]['cos_stmt'].mean()), 3), user_drive=fmt(B_[t]['usr']),
                     change_sharedness=fmt(B_[t]['dshared']) if t != b_ else '', change_sharedness_nb59=p59(t, 'change_sharedness'),
                     change_user_drive=fmt(B_[t]['dusr']) if t != b_ else '', change_user_drive_nb59=p59(t, 'change_user_drive'),
                     change_attack=round(COST[t]['d_asr'], 3), change_user_followed=round(COST[t]['d_uf'], 3)))
VI = np.array(V_ITEMS); RNDV = {}
for f in FAM:
    vals = []
    for sd_ in range(CRIT59['v_seeds']):
        pd_ = RES[f'{f}_random{sd_}']['pairs']; m_ = len(VI); ci_, cs_ = pair_cosine(pd_[:m_])[0], pair_cosine(pd_[m_:])[0]; vals.append(float((ci_ - cs_).mean()))
        rows.append(dict(condition=f'{f}_random{sd_}', label=f'{FAM[f]["label"]} architecture, random weights, draw {sd_} ({m_} items)', sharedness=f'{vals[-1]:+.3f}',
                         cos_instruction=round(float(ci_.mean()), 3), cos_statement=round(float(cs_.mean()), 3)))
    RNDV[f] = vals
DRT = pd.DataFrame(rows); DRT.to_csv(f'{REP}/drive_59b.csv', index=False)
print(f'=== {n} fresh items: sharedness, user drive and their change from the base model, with notebook 59\'s change beside it ==='); print(DRT.to_string(index=False))

# ---- the marks ----
BV = {f: cluster_boot(DR[BASE_OF[f]]['shared'][VI], HOST[VI], NBOOT, SEED) for f in FAM}
V_ = {f: BV[f][0] - max(RNDV[f]) for f in FAM}; V = all(v >= CRIT59['v_min'] for v in V_.values())
C1_FAIL = [t for t in C1_SET if not B_[t]['dshared'][2] < 0]; C1 = not C1_FAIL
C2_TW_FAIL = [t for t in TWIN if not abs(B_[t]['dusr'][0]) <= CRIT59['c2_twin_abs']]
C2_NT_FAIL = [t for t in NOTWIN if not (B_[t]['dusr'][0] <= CRIT59['c2_notwin_max'] and B_[t]['dusr'][2] < 0)]
C2 = not C2_TW_FAIL and not C2_NT_FAIL
x1 = [B_[t]['dshared'][0] for t in H1_SET]; y1 = [COST[t]['d_asr'] for t in H1_SET]; r1, p1 = spearman_perm(x1, y1, NPERM, SEED, strata=[CD[t]['fam'] for t in H1_SET])
x2 = [B_[t]['dusr'][0] for t in H2_SET]; y2 = [COST[t]['d_uf'] for t in H2_SET]; r2, p2 = spearman_perm(x2, y2, NPERM, SEED, strata=[CD[t]['fam'] for t in H2_SET])
h3 = cluster_boot(DR['l_sec']['shared'] - DR['l_sec_user']['shared'], HOST, NBOOT, SEED)
weak = sorted(set(H1_SET) - set(C1_SET))
checks = [
    ('V', f'sharedness of each undefended model on {len(VI)} fresh items minus the highest of its architecture\'s {CRIT59["v_seeds"]} random-weight draws', '; '.join(f'{FAM[f]["label"]} {v:+.3f}' for f, v in V_.items()), f'>= {CRIT59["v_min"]:.2f} on all three', V),
    ('C1', f'every trained defence that cuts attack success by at least 0.15 ({len(C1_SET)}) lowers sharedness: change in sharedness, interval below 0',
     f'{len(C1_SET) - len(C1_FAIL)} of {len(C1_SET)}' + (f'; not: ' + ', '.join(f'{t} {fmt(B_[t]["dshared"])}' for t in C1_FAIL) if C1_FAIL else ''), f'all {len(C1_SET)}', C1),
    ('C2', f'twin-anchored defences ({len(TWIN)}) keep the user drive, change within {CRIT59["c2_twin_abs"]:.2f} of 0; the {len(NOTWIN)} without the anchor lose at least {-CRIT59["c2_notwin_max"]:.2f} with the interval below 0',
     f'twins {len(TWIN) - len(C2_TW_FAIL)} of {len(TWIN)} (range {min(B_[t]["dusr"][0] for t in TWIN):+.3f} to {max(B_[t]["dusr"][0] for t in TWIN):+.3f}); without {len(NOTWIN) - len(C2_NT_FAIL)} of {len(NOTWIN)} '
     f'(range {min(B_[t]["dusr"][0] for t in NOTWIN):+.3f} to {max(B_[t]["dusr"][0] for t in NOTWIN):+.3f})' + (f'; not: {", ".join(C2_TW_FAIL + C2_NT_FAIL)}' if C2_TW_FAIL or C2_NT_FAIL else ''), 'all 16', C2),
    ('reported', 'the weak defence(s) outside C1: change in sharedness', '; '.join(f'{t} {fmt(B_[t]["dshared"])} (attack {COST[t]["d_asr"]:+.3f})' for t in weak), 'reported', True),
    ('reported', 'Spearman rho within families: change in sharedness against change in attack success; change in user drive against change in user instructions followed', f'{r1:+.3f} (p {p1:.4f}); {r2:+.3f} (p {p2:.4f})', 'reported', True),
    ('reported', 'Meta-SecAlign-8B: sharedness with its input role minus with the document in the user turn', fmt(h3), 'reported', True)]
VT = pd.DataFrame([dict(mark=a, criterion=b, value=v, needed=m_, met=bool(ok_)) for a, b, v, m_, ok_ in checks]); VT.to_csv(f'{REP}/go_criteria_59b.csv', index=False)
if not V:
    DEC = ('NOT CONFIRMED: on the fresh items the shared direction does not stand clearly above random weights (' + '; '.join(f'{FAM[f]["label"]} {v:+.3f}' for f, v in V_.items()) + '), so the two patterns cannot be read there.')
elif C1 and C2:
    DEC = (f'CONFIRMED on {n} fresh items (held-out Alpaca instructions in SQuAD passages): every trained defence that cuts attack success by at least 0.15 lowers sharedness ({len(C1_SET)} of {len(C1_SET)}), '
           f'and the twin-anchored defences keep the user drive (change {min(B_[t]["dusr"][0] for t in TWIN):+.3f} to {max(B_[t]["dusr"][0] for t in TWIN):+.3f}) while the {len(NOTWIN)} without the anchor, '
           f'the SecOPD loss alone and Meta-SecAlign, lose {-max(B_[t]["dusr"][0] for t in NOTWIN):.3f} to {-min(B_[t]["dusr"][0] for t in NOTWIN):.3f}. Both patterns of notebook 59 hold on items it never saw.')
else:
    DEC = ('PARTLY CONFIRMED' if (C1 or C2) else 'NOT CONFIRMED') + f' on {n} fresh items: C1 (every working defence lowers sharedness) ' + ('holds' if C1 else f'fails on {", ".join(C1_FAIL)}') + \
          '; C2 (twins keep the user drive, the rest lose it) ' + ('holds' if C2 else f'fails on {", ".join(C2_TW_FAIL + C2_NT_FAIL)}') + '. Only what holds here can be claimed.'
GTXT = f'{DEC}\nV {"pass" if V else "fail"}, C1 {"pass" if C1 else "fail"}, C2 {"pass" if C2 else "fail"}'
open(f'{REP}/go_59b.txt', 'w').write(GTXT + '\n')

# ---- figure ----
TR = [t for t in H1_SET + [t for t in H2_SET if t not in H1_SET]]; colr = lambda t: 'tab:blue' if t in TWIN else 'tab:red'
fig, ax = plt.subplots(1, 3, figsize=(18, 5.2))
for k_, (key, title, line) in enumerate((('dshared', 'change in sharedness (C1)', None), ('dusr', 'change in user drive (C2)', (CRIT59['c2_twin_abs'], CRIT59['c2_notwin_max'])))):
    xs = np.arange(len(TR)); v = np.array([B_[t][key] for t in TR])
    ax[k_].bar(xs, v[:, 0], color=[colr(t) for t in TR], yerr=[v[:, 0] - v[:, 1], v[:, 2] - v[:, 0]], capsize=2)
    ax[k_].set_xticks(xs); ax[k_].set_xticklabels(TR, rotation=60, ha='right', fontsize=7); ax[k_].axhline(0, c='grey', lw=.6); ax[k_].set_title(title + ': blue twins, red without', fontsize=10)
    if line: ax[k_].axhline(line[0], c='tab:blue', ls=':'); ax[k_].axhline(-line[0], c='tab:blue', ls=':'); ax[k_].axhline(line[1], c='tab:red', ls=':')
for key, mk, lab in (('change_sharedness', 'o', 'sharedness'), ('change_user_drive', 's', 'user drive')):
    a59 = [p59(t, key) for t in TR]; a59b = [B_[t]['dshared' if key == 'change_sharedness' else 'dusr'][0] for t in TR]
    ax[2].scatter(a59, a59b, marker=mk, c=[colr(t) for t in TR], label=lab)
lim = [-0.6, 0.1]; ax[2].plot(lim, lim, c='grey', lw=.6); ax[2].set_xlabel('notebook 59 (BIPIA items)'); ax[2].set_ylabel('notebook 59b (fresh items)'); ax[2].legend(fontsize=8); ax[2].set_title('the same changes on the two item sets', fontsize=10)
for a_ in ax: a_.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{FIG}/confirm_59b.png', dpi=150, bbox_inches='tight'); plt.show()
print('\n=== MARKS ==='); print(VT.to_string(index=False)); print('\n' + GTXT)
log_result('nb59b: the two patterns of notebook 59 on fresh items (held-out Alpaca instructions in SQuAD passages)',
           '\n\n'.join([GTXT, 'Marks:\n' + VT.to_string(index=False), 'Per condition:\n' + DRT.to_string(index=False)]), reports_dir=REP)

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True).stdout.strip()
assert branch == "main", f"the repository is on branch {branch!r}, not main: switch to main before committing"
r = subprocess.run(["python", "tools/commit_cell.py", "nb59b (follow-on work): the two patterns notebook 59 found without a mark, tested on fresh items with marks fixed before any reading: notebook 37's held-out Alpaca instructions in SQuAD validation passages, one item per passage, five prompts per item as in notebook 59 (the clean passage; the instruction or an ordinary SQuAD sentence of about the same length on the passage's last line; the same two in the user turn), the same 19 conditions and the same reader (src/shared_drive.py, unchanged); the user direction built per half of the instructions; marks V (sharedness of each undefended model against three random-weight draws of its architecture), C1 (every trained defence that cuts attack success by at least 0.15 lowers sharedness, 15 conditions) and C2 (the ten twin-anchored defences keep the user drive within 0.03, the six without the anchor lose at least 0.10); reports/shared_drive/drive_59b.csv, statements_59b.csv, go_criteria_59b.csv, go_59b.txt, figures/shared_drive/confirm_59b.png"], capture_output=True, text=True)
print(r.stdout); print(r.stderr)